# 📘 Module 8.1 — Lane Detection

**Goal:** Implement a simple lane detection pipeline — one of the fundamental ADAS tasks.

## Lane Detection in ADAS
Lane detection is critical for:
- 🛣️ **Lane Keep Assist (LKA)**
- ⚠️ **Lane Departure Warning (LDW)**
- 🚗 **Autonomous lane following**

---

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## 1. Traditional Approach: Edge Detection + Hough Transform

In [ ]:
# --- Create a synthetic road image ---
def create_road_image(height=480, width=640):
    """Create a synthetic road scene for lane detection."""
    img = np.zeros((height, width, 3), dtype=np.uint8)
    
    # Sky (blue gradient)
    for y in range(height // 2):
        ratio = y / (height // 2)
        img[y, :] = [int(135 + 50*ratio), int(206 - 50*ratio), 235]
    
    # Road (gray)
    for y in range(height // 2, height):
        img[y, :] = [80, 80, 80]
    
    # Lane markings (white lines)
    vanishing_y = height // 2
    vanishing_x = width // 2
    
    # Left lane
    for y in range(vanishing_y, height):
        progress = (y - vanishing_y) / (height - vanishing_y)
        x = int(vanishing_x - progress * 200)
        if 0 <= x < width - 3:
            img[y, x:x+3] = [255, 255, 255]
    
    # Right lane
    for y in range(vanishing_y, height):
        progress = (y - vanishing_y) / (height - vanishing_y)
        x = int(vanishing_x + progress * 200)
        if 0 <= x < width - 3:
            img[y, x:x+3] = [255, 255, 255]
    
    # Center dashed line (yellow)
    for y in range(vanishing_y, height):
        if (y // 20) % 2 == 0:  # Dashed pattern
            progress = (y - vanishing_y) / (height - vanishing_y)
            x = int(vanishing_x)
            if 0 <= x < width - 2:
                img[y, x:x+2] = [0, 255, 255]
    
    # Add some noise
    noise = np.random.randint(0, 15, img.shape, dtype=np.uint8)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    return img

road_img = create_road_image()
plt.figure(figsize=(10, 6))
plt.imshow(road_img)
plt.title('Synthetic Road Image')
plt.axis('off')
plt.show()

In [ ]:
# --- Edge Detection for Lane Finding ---
from scipy import ndimage

# Convert to grayscale
gray = np.mean(road_img, axis=2)

# Sobel edge detection (horizontal and vertical)
sobel_x = ndimage.sobel(gray, axis=1)  # Vertical edges
sobel_y = ndimage.sobel(gray, axis=0)  # Horizontal edges
edges = np.sqrt(sobel_x**2 + sobel_y**2)

# Region of Interest (ROI) — only look at bottom half
roi_mask = np.zeros_like(edges)
roi_mask[edges.shape[0]//2:, :] = 1
edges_roi = edges * roi_mask

# Threshold
lane_pixels = edges_roi > np.percentile(edges_roi[edges_roi > 0], 90)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Grayscale')
axes[1].imshow(edges, cmap='hot')
axes[1].set_title('Edge Detection')
axes[2].imshow(edges_roi, cmap='hot')
axes[2].set_title('ROI Masked')
axes[3].imshow(lane_pixels, cmap='gray')
axes[3].set_title('Lane Pixels')

for ax in axes:
    ax.axis('off')
plt.suptitle('Traditional Lane Detection Pipeline', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Deep Learning Lane Detection

Modern lane detection uses **semantic segmentation** — classify each pixel as lane or not-lane.

In [ ]:
class LaneSegmentationNet(nn.Module):
    """Simple U-Net-like architecture for lane segmentation."""
    
    def __init__(self):
        super().__init__()
        
        # Encoder (downsampling)
        self.enc1 = self._block(3, 32)
        self.enc2 = self._block(32, 64)
        self.enc3 = self._block(64, 128)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Decoder (upsampling)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec3 = self._block(128, 64)  # 64+64 from skip connection
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = self._block(64, 32)
        
        # Output: per-pixel lane classification
        self.output = nn.Conv2d(32, 1, 1)  # Binary: lane or not
    
    def _block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)       # (B, 32, H, W)
        e2 = self.enc2(self.pool(e1))  # (B, 64, H/2, W/2)
        e3 = self.enc3(self.pool(e2))  # (B, 128, H/4, W/4)
        
        # Decoder with skip connections
        d3 = self.dec3(torch.cat([self.up3(e3), e2], dim=1))  # (B, 64, H/2, W/2)
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))  # (B, 32, H, W)
        
        return torch.sigmoid(self.output(d2))  # (B, 1, H, W) — probability map

model = LaneSegmentationNet()
x = torch.randn(2, 3, 256, 512)  # 2 road images
mask = model(x)

print(f"Input: {x.shape}")
print(f"Output (lane mask): {mask.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## ✅ Key Takeaways

1. **Traditional methods** use edge detection + Hough transform (simple but brittle)
2. **Deep learning** uses semantic segmentation (robust to lighting, weather)
3. **U-Net architecture** with skip connections is effective for pixel-level prediction
4. Lane detection enables LKA, LDW, and autonomous lane following
5. Modern approaches also predict lane **type** (solid, dashed) and **curvature**

---
## 📖 Next Steps
➡️ **Next notebook:** [02_sensor_fusion_planning.ipynb](02_sensor_fusion_planning.ipynb) — Sensor fusion and path planning